# 02 — Recalibrate Simple Signal Grid  
## Locked production research candidate: Core / Secondary 2621

This notebook is the cleaned research notebook for the simple SPX VRP signal grid.

### What this notebook does

1. Loads the production feature panel.
2. Loads the completed naked ATM put outcome panel.
3. Builds the complete-date recalibration matrix.
4. Evaluates the locked **Core / Secondary 2621** candidate.
5. Produces reusable audit tables for future validation and sizing work.

### What was intentionally removed

The earlier exploratory search code was removed from the executable path:

- first-pass through fifth-pass grid sweeps
- failed 90% Core search
- pair 144 intermediate tests
- tenor-line fitting
- hybrid line-fitting tests

Those were useful research steps, but 2621 is now locked. This notebook is meant to be smaller, easier to rerun, and easier to extend without carrying dead code.

In [1]:
# Cell 1 — Setup, imports, paths, and locked benchmark metadata

from pathlib import Path
from copy import deepcopy
import json
import math
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
#
# This notebook is intended to run both from the project root and from a
# notebooks/ subfolder. If neither location has a data/ directory, it falls
# back to the project path used during development.
# ---------------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PANEL_PATH = PROCESSED_DIR / "production_feature_panel_v0_1.parquet"
OUTCOME_PATH = PROCESSED_DIR / "naked_atm_put_eod_panel_v0_1.parquet"

# ---------------------------------------------------------------------
# Tenor universe and grouping
# ---------------------------------------------------------------------

PRODUCTION_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

GROUP_TENORS = {
    "front": [9, 12, 15],
    "middle": [18, 21, 24],
    "back": [27, 30, 33],
}

TENOR_GROUP_MAP = {
    tenor: group
    for group, tenors in GROUP_TENORS.items()
    for tenor in tenors
}

# ---------------------------------------------------------------------
# Locked benchmark from research phase
# ---------------------------------------------------------------------
#
# These values were observed when candidate 2621 was selected.
# They are used as a run-to-run sanity check, not as hard assertions.
# If upstream data changes, these numbers may drift.
# ---------------------------------------------------------------------

LOCKED_CANDIDATE_LABEL = "locked_core_secondary_2621"

EXPECTED_2621_BENCHMARK = {
    "combined_trade_count": 577,
    "combined_trade_frequency": 0.334493,
    "combined_win_rate": 0.837088,
    "core_trade_count": 386,
    "core_trade_frequency": 0.223768,
    "core_win_rate": 0.865285,
    "secondary_only_trade_count": 191,
    "secondary_only_trade_frequency": 0.110725,
    "secondary_only_win_rate": 0.780105,
    "core_share_of_combined": 0.668977,
    "combined_avg_pnl_per_day": 382.748481,
    "combined_aggregate_pnl_per_day": 374.389983,
    "combined_worst_pnl_per_day": -5535.611173,
}

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Feature panel path:", FEATURE_PANEL_PATH)
print("Outcome path:", OUTCOME_PATH)
print("Feature panel exists:", FEATURE_PANEL_PATH.exists())
print("Outcome file exists:", OUTCOME_PATH.exists())

Current working directory: C:\Users\patri\vrp_project\notebooks v1
Project root: C:\Users\patri\vrp_project
Feature panel path: C:\Users\patri\vrp_project\data\processed\production_feature_panel_v0_1.parquet
Outcome path: C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet
Feature panel exists: True
Outcome file exists: True


In [2]:
# Cell 2 — Small reusable data-cleaning helpers

def get_col(df: pd.DataFrame, candidates, required=True, label=None):
    """
    Return the first matching column from a candidate list.

    Matching is case-insensitive, but the original DataFrame column name is
    returned. This keeps the notebook robust to small naming changes in source
    files.
    """
    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in df.columns:
            return c

        c_lower = str(c).lower()
        if c_lower in lower_map:
            return lower_map[c_lower]

    if required:
        raise KeyError(
            f"Missing column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None


def parse_project_date_series(s: pd.Series) -> pd.Series:
    """
    Parse project date columns safely.

    Important:
      Numeric yyyymmdd values such as 20190805 must be parsed as calendar
      dates, not as nanoseconds since 1970.
    """
    s = s.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)

    s_str = s.astype("string").str.strip()
    s_str = s_str.str.replace(r"\.0$", "", regex=True)

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    is_yyyymmdd = s_str.str.fullmatch(r"\d{8}", na=False)

    if is_yyyymmdd.any():
        parsed.loc[is_yyyymmdd] = pd.to_datetime(
            s_str.loc[is_yyyymmdd],
            format="%Y%m%d",
            errors="coerce",
        )

    remaining = ~is_yyyymmdd

    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(
            s_str.loc[remaining],
            errors="coerce",
        )

    return parsed


def clean_binary_series(s: pd.Series) -> pd.Series:
    """
    Convert bool / numeric / string binary values into float 0/1.
    Missing or unresolved values stay NaN.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    numeric = pd.to_numeric(s, errors="coerce")

    if numeric.notna().mean() > 0.95:
        return numeric.astype(float)

    return (
        s.astype("string")
        .str.lower()
        .str.strip()
        .map({
            "true": 1.0,
            "false": 0.0,
            "yes": 1.0,
            "no": 0.0,
            "1": 1.0,
            "0": 0.0,
            "1.0": 1.0,
            "0.0": 0.0,
        })
    )


def require_columns(df: pd.DataFrame, required_cols, label="DataFrame"):
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{label} is missing required columns: {missing_cols}")

In [3]:
# Cell 3 — Load and validate the production feature panel

if not FEATURE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing feature panel: {FEATURE_PANEL_PATH}")

features = pd.read_parquet(FEATURE_PANEL_PATH)
features["date"] = pd.to_datetime(features["date"]).dt.normalize()
features = features.sort_values(["date", "tenor"]).reset_index(drop=True)

required_feature_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

require_columns(features, required_feature_cols, label="Feature panel")

features = features[features["tenor"].isin(PRODUCTION_TENORS)].copy()
features["tenor"] = features["tenor"].astype(int)

eligible = features.dropna(
    subset=[
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "implied_variance",
        "forecast_variance",
    ]
).copy()

eligible = eligible.sort_values(["date", "tenor"]).reset_index(drop=True)

dupes = eligible.duplicated(subset=["date", "tenor"]).sum()

print("Loaded feature panel")
print("  Shape:", features.shape)
print("  Date range:", features["date"].min(), "to", features["date"].max())
print("  Tenors:", sorted(features["tenor"].dropna().unique().tolist()))

print("\nEligible feature panel")
print("  Shape:", eligible.shape)
print("  Date range:", eligible["date"].min(), "to", eligible["date"].max())
print("  Tenors:", sorted(eligible["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        eligible[
            eligible.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows found in eligible feature panel.")

# rv21d and rsi14 are market-level filters, so they should be identical
# across all tenors on a given date.
market_filter_counts = (
    eligible
    .groupby("date")[["rv21d", "rsi14"]]
    .nunique(dropna=False)
)

bad_market_filter_dates = market_filter_counts[
    (market_filter_counts["rv21d"] > 1) |
    (market_filter_counts["rsi14"] > 1)
]

print("  Dates with inconsistent rv21d/rsi14 across tenors:", len(bad_market_filter_dates))

if len(bad_market_filter_dates) > 0:
    display(bad_market_filter_dates.head(20))
    raise ValueError("rv21d or rsi14 differs across tenors on the same date.")

display(eligible.head(10))

Loaded feature panel
  Shape: (18099, 11)
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]

Eligible feature panel
  Shape: (15750, 11)
  Date range: 2019-07-10 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Duplicate date/tenor rows: 0
  Dates with inconsistent rv21d/rsi14 across tenors: 0


,date,tenor,tenor_group,spx_close,implied_variance,forecast_variance,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14
0,2019-07-10,9,front,"2,993.070000",0.010209,0.010971,-0.071968,-1.465831,-0.563268,7.712163,67.404430
1,2019-07-10,12,front,"2,993.070000",0.011378,0.009403,0.190606,-1.095112,-0.216897,7.712163,67.404430
2,2019-07-10,15,front,"2,993.070000",0.012079,0.010344,0.155121,-1.128973,-0.216182,7.712163,67.404430
3,2019-07-10,18,middle,"2,993.070000",0.013868,0.009403,0.388501,-0.584906,0.141210,7.712163,67.404430
4,2019-07-10,21,middle,"2,993.070000",0.015711,0.010075,0.444317,-0.401009,0.274458,7.712163,67.404430
5,2019-07-10,24,middle,"2,993.070000",0.016752,0.009991,0.516811,-0.241134,0.377281,7.712163,67.404430
6,2019-07-10,27,back,"2,993.070000",0.016953,0.009926,0.535301,-0.246532,0.381816,7.712163,67.404430
7,2019-07-10,30,back,"2,993.070000",0.017114,0.010344,0.503503,-0.342138,0.323409,7.712163,67.404430
8,2019-07-10,33,back,"2,993.070000",0.017323,0.009831,0.566508,-0.216783,0.392516,7.712163,67.404430
9,2019-07-11,9,front,"2,999.910000",0.010349,0.008859,0.155524,-0.889726,-0.180556,7.676858,68.576313


In [4]:
# Cell 4 — Load naked ATM put outcomes and build the completed research panel

if not OUTCOME_PATH.exists():
    raise FileNotFoundError(f"Missing outcome file: {OUTCOME_PATH}")

outcomes_raw = pd.read_parquet(OUTCOME_PATH)

# ---------------------------------------------------------------------
# Column mapping
# ---------------------------------------------------------------------

date_col = get_col(
    outcomes_raw,
    ["trade_dt", "trade_date", "date", "timestamp", "datetime"],
    label="date",
)

tenor_col = get_col(
    outcomes_raw,
    ["entry_tenor", "tenor", "target_tenor", "actual_dte", "dte"],
    label="tenor",
)

win_col = get_col(
    outcomes_raw,
    ["win_clean", "win", "test_win"],
    label="win",
)

expired_otm_col = get_col(
    outcomes_raw,
    ["expired_otm_clean", "expired_otm"],
    required=False,
    label="expired_otm",
)

pnl_dollars_col = get_col(
    outcomes_raw,
    ["normalized_pnl_dollars", "test_pnl", "sized_pnl"],
    required=False,
    label="normalized_pnl_dollars",
)

pnl_pct_col = get_col(
    outcomes_raw,
    ["normalized_pnl_pct"],
    required=False,
    label="normalized_pnl_pct",
)

actual_dte_col = get_col(
    outcomes_raw,
    ["actual_dte"],
    required=False,
    label="actual_dte",
)

expiry_spx_col = get_col(
    outcomes_raw,
    ["expiry_spx_close"],
    required=False,
    label="expiry_spx_close",
)

entry_credit_col = get_col(
    outcomes_raw,
    ["entry_credit_points", "atm_put_mid"],
    required=False,
    label="entry_credit_points",
)

short_pnl_points_col = get_col(
    outcomes_raw,
    ["short_option_pnl_points"],
    required=False,
    label="short_option_pnl_points",
)

print("Outcome column mapping:")
print({
    "date_col": date_col,
    "tenor_col": tenor_col,
    "win_col": win_col,
    "expired_otm_col": expired_otm_col,
    "pnl_dollars_col": pnl_dollars_col,
    "pnl_pct_col": pnl_pct_col,
    "actual_dte_col": actual_dte_col,
    "expiry_spx_col": expiry_spx_col,
    "entry_credit_col": entry_credit_col,
    "short_pnl_points_col": short_pnl_points_col,
})

# ---------------------------------------------------------------------
# Standardize outcomes
# ---------------------------------------------------------------------

outcomes = pd.DataFrame({
    "date": parse_project_date_series(outcomes_raw[date_col]).dt.normalize(),
    "tenor": pd.to_numeric(outcomes_raw[tenor_col], errors="coerce").astype("Int64"),
    "win_clean": clean_binary_series(outcomes_raw[win_col]),
})

if expired_otm_col is not None:
    outcomes["expired_otm_clean"] = clean_binary_series(outcomes_raw[expired_otm_col])
else:
    outcomes["expired_otm_clean"] = np.nan

if pnl_dollars_col is not None:
    outcomes["normalized_pnl_dollars"] = pd.to_numeric(outcomes_raw[pnl_dollars_col], errors="coerce")
else:
    outcomes["normalized_pnl_dollars"] = np.nan

if pnl_pct_col is not None:
    outcomes["normalized_pnl_pct"] = pd.to_numeric(outcomes_raw[pnl_pct_col], errors="coerce")
else:
    outcomes["normalized_pnl_pct"] = np.nan

if actual_dte_col is not None:
    outcomes["actual_dte"] = pd.to_numeric(outcomes_raw[actual_dte_col], errors="coerce")
else:
    outcomes["actual_dte"] = np.nan

if expiry_spx_col is not None:
    outcomes["expiry_spx_close"] = pd.to_numeric(outcomes_raw[expiry_spx_col], errors="coerce")
else:
    outcomes["expiry_spx_close"] = np.nan

if entry_credit_col is not None:
    outcomes["entry_credit_points"] = pd.to_numeric(outcomes_raw[entry_credit_col], errors="coerce")
else:
    outcomes["entry_credit_points"] = np.nan

if short_pnl_points_col is not None:
    outcomes["short_option_pnl_points"] = pd.to_numeric(outcomes_raw[short_pnl_points_col], errors="coerce")
else:
    outcomes["short_option_pnl_points"] = np.nan

outcomes = outcomes[outcomes["tenor"].isin(PRODUCTION_TENORS)].copy()
outcomes = outcomes.dropna(subset=["date", "tenor"]).copy()
outcomes["tenor"] = outcomes["tenor"].astype(int)

dupes = outcomes.duplicated(subset=["date", "tenor"]).sum()

print("\nStandardized outcome panel")
print("  Shape:", outcomes.shape)
print("  Date range:", outcomes["date"].min(), "to", outcomes["date"].max())
print("  Tenors:", sorted(outcomes["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        outcomes[
            outcomes.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Outcome panel has duplicate date/tenor rows.")

# ---------------------------------------------------------------------
# Join outcomes to eligible features.
# ---------------------------------------------------------------------
#
# Missing outcomes near the right edge are expected because later-dated
# longer-tenor trades may not have expired yet. Those rows are dropped.
#
# Missing outcomes before the last completed outcome date for that tenor are
# treated as a data hole and raise an error.
# ---------------------------------------------------------------------

research_panel_raw = eligible.merge(
    outcomes,
    on=["date", "tenor"],
    how="left",
)

completed_outcomes = outcomes.dropna(subset=["win_clean"]).copy()

last_completed_date_by_tenor = (
    completed_outcomes
    .groupby("tenor")["date"]
    .max()
    .rename("last_completed_outcome_date")
    .reset_index()
)

missing_outcomes = (
    research_panel_raw[research_panel_raw["win_clean"].isna()]
    [["date", "tenor", "tenor_group", "spx_close", "vrp_log", "vrp_z_3m", "vrp_z_1y", "rv21d", "rsi14"]]
    .copy()
)

missing_outcomes = missing_outcomes.merge(
    last_completed_date_by_tenor,
    on="tenor",
    how="left",
)

unexpected_missing = missing_outcomes[
    missing_outcomes["date"] <= missing_outcomes["last_completed_outcome_date"]
].copy()

print("\nResearch panel after outcome join")
print("  Shape:", research_panel_raw.shape)
print("  Rows with missing win_clean:", int(research_panel_raw["win_clean"].isna().sum()))

print("\nLast completed outcome date by tenor:")
display(last_completed_date_by_tenor)

if len(missing_outcomes) > 0:
    missing_summary = (
        missing_outcomes
        .groupby("tenor")
        .agg(
            missing_rows=("date", "size"),
            first_missing_date=("date", "min"),
            last_missing_date=("date", "max"),
            last_completed_outcome_date=("last_completed_outcome_date", "max"),
        )
        .reset_index()
    )

    print("\nMissing outcome summary by tenor:")
    display(missing_summary)

if len(unexpected_missing) > 0:
    print("\nUnexpected missing outcomes found before/equal to last completed date for that tenor:")
    display(unexpected_missing.head(30))
    raise ValueError("Unexpected non-right-edge missing outcomes found.")

print("\nRight-edge censored rows to drop:", len(missing_outcomes))

research_panel = research_panel_raw.dropna(subset=["win_clean"]).copy()
research_panel = research_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

required_outcome_cols = ["win_clean", "normalized_pnl_dollars", "actual_dte"]
require_columns(research_panel, required_outcome_cols, label="Completed research panel")

if research_panel[required_outcome_cols].isna().any().any():
    display(research_panel[research_panel[required_outcome_cols].isna().any(axis=1)].head(20))
    raise ValueError("Completed research panel has missing win/P&L/actual_dte values.")

print("\nFinal completed research panel")
print("  Shape:", research_panel.shape)
print("  Date range:", research_panel["date"].min(), "to", research_panel["date"].max())
print("  Tenors:", sorted(research_panel["tenor"].dropna().unique().tolist()))
print("  Overall win rate:", research_panel["win_clean"].mean())

outcome_by_tenor = (
    research_panel
    .groupby("tenor")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_pct=("normalized_pnl_pct", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
    )
    .reset_index()
)

print("\nOutcome summary by tenor:")
display(outcome_by_tenor)

Outcome column mapping:
{'date_col': 'trade_dt', 'tenor_col': 'entry_tenor', 'win_col': 'win', 'expired_otm_col': 'expired_otm', 'pnl_dollars_col': 'normalized_pnl_dollars', 'pnl_pct_col': None, 'actual_dte_col': 'actual_dte', 'expiry_spx_col': 'expiry_spx_close', 'entry_credit_col': 'entry_credit_points', 'short_pnl_points_col': 'short_option_pnl_points'}

Standardized outcome panel
  Shape: (18099, 10)
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Duplicate date/tenor rows: 0

Research panel after outcome join
  Shape: (15750, 19)
  Rows with missing win_clean: 153

Last completed outcome date by tenor:


,tenor,last_completed_outcome_date
0,9,2026-06-12
1,12,2026-06-09
2,15,2026-06-05
3,18,2026-06-03
4,21,2026-05-29
5,24,2026-05-28
6,27,2026-05-22
7,30,2026-05-22
8,33,2026-05-19



Missing outcome summary by tenor:


,tenor,missing_rows,first_missing_date,last_missing_date,last_completed_outcome_date
0,9,8,2026-06-15,2026-06-25,2026-06-12
1,12,11,2026-06-10,2026-06-25,2026-06-09
2,15,13,2026-06-08,2026-06-25,2026-06-05
3,18,15,2026-06-04,2026-06-25,2026-06-03
4,21,18,2026-06-01,2026-06-25,2026-05-29
5,24,19,2026-05-29,2026-06-25,2026-05-28
6,27,22,2026-05-26,2026-06-25,2026-05-22
7,30,22,2026-05-26,2026-06-25,2026-05-22
8,33,25,2026-05-20,2026-06-25,2026-05-19



Right-edge censored rows to drop: 153

Final completed research panel
  Shape: (15597, 19)
  Date range: 2019-07-10 00:00:00 to 2026-06-12 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Overall win rate: 0.7633519266525614

Outcome summary by tenor:


,tenor,rows,win_rate,avg_pnl_dollars,avg_pnl_pct,avg_actual_dte
0,9,1742,0.731343,"1,123.741635",NaN,8.940299
1,12,1739,0.742381,"1,816.837565",NaN,11.848189
2,15,1737,0.748417,"2,462.510510",NaN,15.940127
3,18,1735,0.753890,"2,648.826727",NaN,17.438040
4,21,1732,0.760393,"3,264.124558",NaN,21.727483
5,24,1731,0.767187,"3,535.662513",NaN,23.038128
6,27,1728,0.785301,"4,430.317093",NaN,27.284722
7,30,1728,0.788773,"4,900.713063",NaN,29.941551
8,33,1725,0.793043,"5,573.755492",NaN,32.848116


In [5]:
# Cell 5 — Build complete-date sweep panel and matrix representation

# ---------------------------------------------------------------------
# Historical win probability by tenor
# ---------------------------------------------------------------------
#
# Selection rule:
#   When more than one tenor qualifies on the same date, select the qualifying
#   tenor with the highest full-sample historical win probability.
#
# This is the same selection priority used during the research process.
# ---------------------------------------------------------------------

tenor_win_probability = (
    research_panel
    .groupby("tenor")
    .agg(
        historical_trades=("win_clean", "size"),
        historical_win_probability=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
    )
    .reset_index()
    .sort_values("tenor")
)

tenor_win_prob_map = dict(
    zip(
        tenor_win_probability["tenor"].astype(int),
        tenor_win_probability["historical_win_probability"].astype(float),
    )
)

research_panel = research_panel.copy()
research_panel["tenor_historical_win_probability"] = (
    research_panel["tenor"].astype(int).map(tenor_win_prob_map)
)

print("Historical win probability by tenor:")
display(tenor_win_probability)

# ---------------------------------------------------------------------
# Sweep panel
# ---------------------------------------------------------------------

sweep_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "tenor_historical_win_probability",
]

sweep_panel = research_panel[sweep_cols].copy()

sweep_panel["date"] = pd.to_datetime(sweep_panel["date"]).dt.normalize()
sweep_panel["tenor"] = sweep_panel["tenor"].astype(int)
sweep_panel["tenor_group"] = sweep_panel["tenor_group"].astype(str)

numeric_cols = [
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "tenor_historical_win_probability",
]

for col in numeric_cols:
    sweep_panel[col] = pd.to_numeric(sweep_panel[col], errors="coerce")

before_drop = len(sweep_panel)

sweep_panel = sweep_panel.dropna(
    subset=[
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "tenor_historical_win_probability",
    ]
).copy()

after_drop = len(sweep_panel)

print("\nSweep panel row cleaning")
print("  Rows before drop:", before_drop)
print("  Rows after drop: ", after_drop)
print("  Rows dropped:     ", before_drop - after_drop)

dupes = sweep_panel.duplicated(subset=["date", "tenor"]).sum()
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        sweep_panel[
            sweep_panel.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows in sweep_panel.")

# ---------------------------------------------------------------------
# Complete-date filter
# ---------------------------------------------------------------------
#
# Keep only dates where all 9 tenors have completed outcomes.
# This keeps trade-frequency comparisons clean.
# ---------------------------------------------------------------------

REQUIRED_TENOR_COUNT = len(PRODUCTION_TENORS)

tenor_count_by_date = (
    sweep_panel
    .groupby("date")["tenor"]
    .nunique()
    .rename("tenor_count")
)

complete_dates = tenor_count_by_date[tenor_count_by_date == REQUIRED_TENOR_COUNT].index
incomplete_dates = tenor_count_by_date[tenor_count_by_date != REQUIRED_TENOR_COUNT]

print("\nComplete-date filter")
print("  Original sweep rows:", len(sweep_panel))
print("  Original sweep dates:", sweep_panel["date"].nunique())
print("  Complete dates:", len(complete_dates))
print("  Incomplete dates:", len(incomplete_dates))

if len(incomplete_dates) > 0:
    print("\nIncomplete date summary:")
    display(
        incomplete_dates
        .reset_index()
        .sort_values("date")
    )

sweep_panel = sweep_panel[sweep_panel["date"].isin(complete_dates)].copy()
sweep_panel = sweep_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

sweep_dates = pd.Index(sorted(sweep_panel["date"].unique()))
num_sweep_dates = len(sweep_dates)
TRADE_FREQUENCY_DENOMINATOR = num_sweep_dates

print("\nClean sweep panel")
print("  Rows:", len(sweep_panel))
print("  Dates:", num_sweep_dates)
print("  Date range:", sweep_dates.min(), "to", sweep_dates.max())
print("  Trade frequency denominator:", TRADE_FREQUENCY_DENOMINATOR)

clean_tenor_count_by_date = sweep_panel.groupby("date")["tenor"].nunique()

if not (clean_tenor_count_by_date == REQUIRED_TENOR_COUNT).all():
    raise ValueError("Some remaining dates do not have all 9 tenors.")

print("\nRows by tenor group after complete-date filter:")
display(
    sweep_panel
    .groupby("tenor_group")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_vrp=("vrp_log", "mean"),
        avg_z3m=("vrp_z_3m", "mean"),
        avg_z1y=("vrp_z_1y", "mean"),
        avg_rv21d=("rv21d", "mean"),
        avg_rsi14=("rsi14", "mean"),
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Matrix representation
# ---------------------------------------------------------------------
#
# All strategy logic below uses matrix arrays:
#   rows = dates
#   columns = tenors [9, 12, ..., 33]
# ---------------------------------------------------------------------

TENOR_ARRAY = np.array(PRODUCTION_TENORS, dtype=int)
TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

GROUP_COLS = {
    group: [TENOR_TO_COL[t] for t in tenors]
    for group, tenors in GROUP_TENORS.items()
}

def pivot_to_matrix(value_col):
    mat_df = (
        sweep_panel
        .pivot(index="date", columns="tenor", values=value_col)
        .reindex(index=sweep_dates, columns=PRODUCTION_TENORS)
    )

    if mat_df.isna().any().any():
        bad = mat_df[mat_df.isna().any(axis=1)].head(10)
        display(bad)
        raise ValueError(f"Matrix for {value_col} contains missing values.")

    return mat_df.to_numpy()

spx_mat = pivot_to_matrix("spx_close")
vrp_mat = pivot_to_matrix("vrp_log")
z3m_mat = pivot_to_matrix("vrp_z_3m")
z1y_mat = pivot_to_matrix("vrp_z_1y")
win_mat = pivot_to_matrix("win_clean")
pnl_mat = pivot_to_matrix("normalized_pnl_dollars")
actual_dte_mat = pivot_to_matrix("actual_dte")
tenor_win_prob_mat = pivot_to_matrix("tenor_historical_win_probability")

rv21d_by_date = (
    sweep_panel.groupby("date")["rv21d"].first().reindex(sweep_dates).to_numpy()
)

rsi_by_date = (
    sweep_panel.groupby("date")["rsi14"].first().reindex(sweep_dates).to_numpy()
)

with np.errstate(divide="ignore", invalid="ignore"):
    pnl_per_day_mat = pnl_mat / actual_dte_mat

pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

# Highest historical tenor win probability gets first priority.
selection_priority = (
    tenor_win_probability
    .sort_values(
        ["historical_win_probability", "tenor"],
        ascending=[False, False],
    )
    [["tenor", "historical_win_probability"]]
    .reset_index(drop=True)
)

PRIORITY_COLS = [
    TENOR_TO_COL[int(t)]
    for t in selection_priority["tenor"].tolist()
]

print("\nSelection priority by historical tenor win probability:")
display(selection_priority)

print("\nPriority tenor order:", [int(TENOR_ARRAY[c]) for c in PRIORITY_COLS])

Historical win probability by tenor:


,tenor,historical_trades,historical_win_probability,avg_pnl_dollars,avg_actual_dte
0,9,1742,0.731343,"1,123.741635",8.940299
1,12,1739,0.742381,"1,816.837565",11.848189
2,15,1737,0.748417,"2,462.510510",15.940127
3,18,1735,0.753890,"2,648.826727",17.438040
4,21,1732,0.760393,"3,264.124558",21.727483
5,24,1731,0.767187,"3,535.662513",23.038128
6,27,1728,0.785301,"4,430.317093",27.284722
7,30,1728,0.788773,"4,900.713063",29.941551
8,33,1725,0.793043,"5,573.755492",32.848116



Sweep panel row cleaning
  Rows before drop: 15597
  Rows after drop:  15597
  Rows dropped:      0
  Duplicate date/tenor rows: 0

Complete-date filter
  Original sweep rows: 15597
  Original sweep dates: 1742
  Complete dates: 1725
  Incomplete dates: 17

Incomplete date summary:


,date,tenor_count
0,2026-05-20,8
1,2026-05-21,8
2,2026-05-22,8
3,2026-05-26,6
4,2026-05-27,6
5,2026-05-28,6
6,2026-05-29,5
7,2026-06-01,4
8,2026-06-02,4
9,2026-06-03,4



Clean sweep panel
  Rows: 15525
  Dates: 1725
  Date range: 2019-07-10 00:00:00 to 2026-05-19 00:00:00
  Trade frequency denominator: 1725

Rows by tenor group after complete-date filter:


,tenor_group,rows,win_rate,avg_vrp,avg_z3m,avg_z1y,avg_rv21d,avg_rsi14
0,back,5175,0.788792,0.508307,0.068748,0.076143,16.720316,55.807832
1,front,5175,0.743188,0.667672,0.071348,0.087078,16.720316,55.807832
2,middle,5175,0.760580,0.524071,0.068264,0.079054,16.720316,55.807832



Selection priority by historical tenor win probability:


,tenor,historical_win_probability
0,33,0.793043
1,30,0.788773
2,27,0.785301
3,24,0.767187
4,21,0.760393
5,18,0.753890
6,15,0.748417
7,12,0.742381
8,9,0.731343



Priority tenor order: [33, 30, 27, 24, 21, 18, 15, 12, 9]


## Locked 2621 parameters

Candidate **2621** is the locked research winner.

### Rules

- Use front / middle / back tenor buckets.
- Do **not** fit a line across the 9 tenors.
- Check **Core first**.
- If Core qualifies, take the highest-priority qualifying tenor.
- If Core does not qualify, check **Secondary**.
- If Secondary qualifies, take the highest-priority qualifying tenor.
- Maximum one selected trade per date.
- Trade frequency denominator is the number of complete-date rows.

### Locked thresholds

| Layer | Parameter | Front | Middle | Back |
|---|---|---:|---:|---:|
| Core | VRP | 0.60 | 0.65 | 0.70 |
| Core | z3m | 0.55 | 0.75 | 0.75 |
| Core | z1y | 0.65 | 0.65 | 0.75 |
| Core | RSI cap | 70 | 68 | 66 |
| Core | RV21D floor | 8.5 | 8.5 | 8.5 |
| Secondary | VRP | 0.60 | 0.60 | 0.70 |
| Secondary | z3m | 0.50 | 0.50 | 0.50 |
| Secondary | z1y | 0.40 | 0.50 | 0.50 |
| Secondary | RSI cap | 74 | 70 | 68 |
| Secondary | RV21D floor | 6.5 | 6.5 | 6.5 |

In [6]:
# Cell 6 — Locked 2621 parameter dictionary

LOCKED_2621 = {
    "candidate_label": LOCKED_CANDIDATE_LABEL,

    "core": {
        "front": {
            "vrp": 0.60,
            "z3m": 0.55,
            "z1y": 0.65,
            "rsi_cap": 70.0,
            "rv21d_floor": 8.5,
        },
        "middle": {
            "vrp": 0.65,
            "z3m": 0.75,
            "z1y": 0.65,
            "rsi_cap": 68.0,
            "rv21d_floor": 8.5,
        },
        "back": {
            "vrp": 0.70,
            "z3m": 0.75,
            "z1y": 0.75,
            "rsi_cap": 66.0,
            "rv21d_floor": 8.5,
        },
    },

    "secondary": {
        "front": {
            "vrp": 0.60,
            "z3m": 0.50,
            "z1y": 0.40,
            "rsi_cap": 74.0,
            "rv21d_floor": 6.5,
        },
        "middle": {
            "vrp": 0.60,
            "z3m": 0.50,
            "z1y": 0.50,
            "rsi_cap": 70.0,
            "rv21d_floor": 6.5,
        },
        "back": {
            "vrp": 0.70,
            "z3m": 0.50,
            "z1y": 0.50,
            "rsi_cap": 68.0,
            "rv21d_floor": 6.5,
        },
    },
}


def candidate_params_to_table(candidate_params):
    rows = []

    for layer in ["core", "secondary"]:
        for group in ["front", "middle", "back"]:
            p = candidate_params[layer][group]

            rows.append({
                "candidate": candidate_params.get("candidate_label", ""),
                "layer": layer,
                "tenor_group": group,
                "tenors": ", ".join(map(str, GROUP_TENORS[group])),
                "vrp_threshold": p["vrp"],
                "z3m_threshold": p["z3m"],
                "z1y_threshold": p["z1y"],
                "rsi_cap": p["rsi_cap"],
                "rv21d_floor": p["rv21d_floor"],
            })

    return pd.DataFrame(rows)


locked_2621_parameter_table = candidate_params_to_table(LOCKED_2621)

display(locked_2621_parameter_table)

,candidate,layer,tenor_group,tenors,vrp_threshold,z3m_threshold,z1y_threshold,rsi_cap,rv21d_floor
0,locked_core_secondary_2621,core,front,"9, 12, 15",0.600000,0.550000,0.650000,70.000000,8.500000
1,locked_core_secondary_2621,core,middle,"18, 21, 24",0.650000,0.750000,0.650000,68.000000,8.500000
2,locked_core_secondary_2621,core,back,"27, 30, 33",0.700000,0.750000,0.750000,66.000000,8.500000
3,locked_core_secondary_2621,secondary,front,"9, 12, 15",0.600000,0.500000,0.400000,74.000000,6.500000
4,locked_core_secondary_2621,secondary,middle,"18, 21, 24",0.600000,0.500000,0.500000,70.000000,6.500000
5,locked_core_secondary_2621,secondary,back,"27, 30, 33",0.700000,0.500000,0.500000,68.000000,6.500000


In [7]:
# Cell 7 — Core/Secondary selection and evaluation utilities
#
# These functions are intentionally reusable. Future tests should modify the
# parameter dictionary and call evaluate_core_secondary_candidate().
#
# No line-fitting or exploratory grid-search code lives in this notebook.

REQUIRED_LAYER_KEYS = ["vrp", "z3m", "z1y", "rsi_cap", "rv21d_floor"]


def validate_layer_params(layer_params, layer_name):
    for group in ["front", "middle", "back"]:
        if group not in layer_params:
            raise ValueError(f"{layer_name} missing group: {group}")

        missing = [k for k in REQUIRED_LAYER_KEYS if k not in layer_params[group]]
        if missing:
            raise ValueError(f"{layer_name}.{group} missing keys: {missing}")


def validate_core_secondary_candidate(candidate_params):
    """
    Validate basic candidate shape and monotonic strictness.

    Core must be stricter than or equal to Secondary:
      - VRP, z3m, z1y, RV floors: Core >= Secondary
      - RSI cap: Core <= Secondary
    """
    validate_layer_params(candidate_params["core"], "core")
    validate_layer_params(candidate_params["secondary"], "secondary")

    for group in ["front", "middle", "back"]:
        core = candidate_params["core"][group]
        secondary = candidate_params["secondary"][group]

        for key in ["vrp", "z3m", "z1y", "rv21d_floor"]:
            if core[key] < secondary[key]:
                raise ValueError(
                    f"Core must be stricter than Secondary for {group}.{key}. "
                    f"Core={core[key]}, Secondary={secondary[key]}"
                )

        if core["rsi_cap"] > secondary["rsi_cap"]:
            raise ValueError(
                f"Core RSI cap must be lower/tighter than Secondary for {group}. "
                f"Core={core['rsi_cap']}, Secondary={secondary['rsi_cap']}"
            )


def build_selected_cols_for_layer(layer_params):
    """
    Build selected tenor column by date for a single layer.

    Return:
      selected_col_by_date:
        int array length num_sweep_dates
        -1 means no qualifying tenor
        otherwise column index into TENOR_ARRAY
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = TENOR_GROUP_MAP[int(tenor)]
        p = layer_params[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(p["vrp"])) &
            (z3m_mat[:, col] >= float(p["z3m"])) &
            (z1y_mat[:, col] >= float(p["z1y"])) &
            (rsi_by_date <= float(p["rsi_cap"])) &
            (rv21d_by_date >= float(p["rv21d_floor"]))
        )

    selected_col_by_date = np.full(num_sweep_dates, -1, dtype=np.int8)

    for col in PRIORITY_COLS:
        take = (selected_col_by_date == -1) & qualifies[:, col]
        selected_col_by_date[take] = col

    return selected_col_by_date


def summarize_selected_cols(selected_col_by_date):
    """
    Summary metrics for a selected-col array.
    """
    trade_mask = selected_col_by_date >= 0
    trade_count = int(trade_mask.sum())

    selected_tenor_counts_empty = {
        f"selected_tenor_{int(t)}_count": 0
        for t in TENOR_ARRAY
    }

    if trade_count == 0:
        return {
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "worst_pnl_per_day": np.nan,
            "avg_actual_dte": np.nan,
            "avg_selected_tenor": np.nan,
            "front_count": 0,
            "middle_count": 0,
            "back_count": 0,
            **selected_tenor_counts_empty,
        }

    row_idx = np.where(trade_mask)[0]
    col_idx = selected_col_by_date[trade_mask].astype(int)

    selected_wins = win_mat[row_idx, col_idx]
    selected_pnls = pnl_mat[row_idx, col_idx]
    selected_dtes = actual_dte_mat[row_idx, col_idx]
    selected_pnl_per_day = pnl_per_day_mat[row_idx, col_idx]
    selected_tenors = TENOR_ARRAY[col_idx]

    selected_tenor_counts = {
        f"selected_tenor_{int(t)}_count": int((selected_tenors == t).sum())
        for t in TENOR_ARRAY
    }

    return {
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(np.nanmean(selected_wins)),

        "avg_pnl_per_day": float(np.nanmean(selected_pnl_per_day)),
        "median_pnl_per_day": float(np.nanmedian(selected_pnl_per_day)),
        "aggregate_pnl_per_day": float(np.nansum(selected_pnls) / np.nansum(selected_dtes)),
        "worst_pnl_per_day": float(np.nanmin(selected_pnl_per_day)),

        "avg_actual_dte": float(np.nanmean(selected_dtes)),
        "avg_selected_tenor": float(np.nanmean(selected_tenors)),

        "front_count": int(np.isin(selected_tenors, GROUP_TENORS["front"]).sum()),
        "middle_count": int(np.isin(selected_tenors, GROUP_TENORS["middle"]).sum()),
        "back_count": int(np.isin(selected_tenors, GROUP_TENORS["back"]).sum()),

        **selected_tenor_counts,
    }


def combine_core_secondary(core_selected, secondary_raw_selected):
    """
    Core gets first priority. Secondary only fills dates where Core does not trade.
    """
    layer = np.full(num_sweep_dates, "None", dtype=object)

    final_selected = np.where(
        core_selected >= 0,
        core_selected,
        secondary_raw_selected,
    ).astype(np.int8)

    layer[core_selected >= 0] = "Core"

    secondary_only_mask = (core_selected < 0) & (secondary_raw_selected >= 0)
    layer[secondary_only_mask] = "Secondary"

    secondary_only_selected = np.where(
        secondary_only_mask,
        secondary_raw_selected,
        -1,
    ).astype(np.int8)

    return final_selected, secondary_only_selected, layer


def selected_cols_to_trades(selected_col_by_date, layer_by_date=None, candidate_label="candidate"):
    """
    Convert selected columns into a trade-level audit DataFrame.
    """
    trade_mask = selected_col_by_date >= 0

    if layer_by_date is None:
        layer_by_date = np.full(num_sweep_dates, "Selected", dtype=object)

    row_idx = np.where(trade_mask)[0]
    col_idx = selected_col_by_date[trade_mask].astype(int)

    trades = pd.DataFrame({
        "candidate": candidate_label,
        "date": sweep_dates[row_idx],
        "layer": layer_by_date[trade_mask],
        "selected_col": col_idx,
        "selected_tenor": TENOR_ARRAY[col_idx],
        "tenor_group": [TENOR_GROUP_MAP[int(t)] for t in TENOR_ARRAY[col_idx]],

        "spx_close": spx_mat[row_idx, col_idx],
        "rv21d": rv21d_by_date[row_idx],
        "rsi14": rsi_by_date[row_idx],

        "vrp_log": vrp_mat[row_idx, col_idx],
        "vrp_z_3m": z3m_mat[row_idx, col_idx],
        "vrp_z_1y": z1y_mat[row_idx, col_idx],

        "tenor_historical_win_probability": tenor_win_prob_mat[row_idx, col_idx],
        "win": win_mat[row_idx, col_idx],
        "normalized_pnl_dollars": pnl_mat[row_idx, col_idx],
        "actual_dte": actual_dte_mat[row_idx, col_idx],
        "pnl_per_day": pnl_per_day_mat[row_idx, col_idx],
    })

    trades = trades.sort_values("date").reset_index(drop=True)
    trades["trade_sequence"] = np.arange(1, len(trades) + 1)

    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()

    return trades


def summarize_trade_frame(df, label=None, denominator=TRADE_FREQUENCY_DENOMINATOR):
    """
    Summary metrics for any trade-level subset.
    """
    n = len(df)

    if n == 0:
        return {
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        }

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_dte = float(df["actual_dte"].sum())

    # Drawdown is meaningful only for date-ordered full paths.
    path = df.sort_values("date").copy()
    cum_pnl = path["normalized_pnl_dollars"].cumsum()
    drawdown = cum_pnl - cum_pnl.cummax()
    rolling_20 = path["normalized_pnl_dollars"].rolling(20).sum()

    return {
        "label": label,
        "trade_count": int(n),
        "trade_frequency": float(n / denominator),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_dte) if total_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(drawdown.min()),
        "worst_20_trade_sum": float(rolling_20.min()) if rolling_20.notna().any() else np.nan,
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    }


def evaluate_core_secondary_candidate(candidate_params, candidate_label=None):
    """
    Evaluate a Core/Secondary candidate.

    Returns:
      result_summary: one-row DataFrame-friendly dict
      selected_outputs: selected arrays
      trades: selected trade DataFrame with Core / Secondary layer labels
    """
    validate_core_secondary_candidate(candidate_params)

    if candidate_label is None:
        candidate_label = candidate_params.get("candidate_label", "candidate")

    core_selected = build_selected_cols_for_layer(candidate_params["core"])
    secondary_raw_selected = build_selected_cols_for_layer(candidate_params["secondary"])

    combined_selected, secondary_only_selected, layer_by_date = combine_core_secondary(
        core_selected,
        secondary_raw_selected,
    )

    core_metrics = summarize_selected_cols(core_selected)
    secondary_only_metrics = summarize_selected_cols(secondary_only_selected)
    combined_metrics = summarize_selected_cols(combined_selected)

    combined_trade_count = combined_metrics["trade_count"]

    if combined_trade_count > 0:
        core_share = core_metrics["trade_count"] / combined_trade_count
        secondary_share = secondary_only_metrics["trade_count"] / combined_trade_count
    else:
        core_share = np.nan
        secondary_share = np.nan

    result = {
        "candidate": candidate_label,

        "core_trade_count": int(core_metrics["trade_count"]),
        "core_trade_frequency": float(core_metrics["trade_frequency"]),
        "core_win_rate": float(core_metrics["win_rate"]),
        "core_avg_pnl_per_day": float(core_metrics["avg_pnl_per_day"]),
        "core_aggregate_pnl_per_day": float(core_metrics["aggregate_pnl_per_day"]),
        "core_worst_pnl_per_day": float(core_metrics["worst_pnl_per_day"]),

        "secondary_only_trade_count": int(secondary_only_metrics["trade_count"]),
        "secondary_only_trade_frequency": float(secondary_only_metrics["trade_frequency"]),
        "secondary_only_win_rate": float(secondary_only_metrics["win_rate"]),
        "secondary_only_avg_pnl_per_day": float(secondary_only_metrics["avg_pnl_per_day"]),
        "secondary_only_aggregate_pnl_per_day": float(secondary_only_metrics["aggregate_pnl_per_day"]),
        "secondary_only_worst_pnl_per_day": float(secondary_only_metrics["worst_pnl_per_day"]),

        "combined_trade_count": int(combined_metrics["trade_count"]),
        "combined_trade_frequency": float(combined_metrics["trade_frequency"]),
        "combined_win_rate": float(combined_metrics["win_rate"]),
        "combined_avg_pnl_per_day": float(combined_metrics["avg_pnl_per_day"]),
        "combined_median_pnl_per_day": float(combined_metrics["median_pnl_per_day"]),
        "combined_aggregate_pnl_per_day": float(combined_metrics["aggregate_pnl_per_day"]),
        "combined_worst_pnl_per_day": float(combined_metrics["worst_pnl_per_day"]),
        "combined_avg_actual_dte": float(combined_metrics["avg_actual_dte"]),
        "combined_avg_selected_tenor": float(combined_metrics["avg_selected_tenor"]),

        "core_share_of_combined": float(core_share),
        "secondary_share_of_combined": float(secondary_share),

        "combined_front_count": int(combined_metrics["front_count"]),
        "combined_middle_count": int(combined_metrics["middle_count"]),
        "combined_back_count": int(combined_metrics["back_count"]),
    }

    for tenor in TENOR_ARRAY:
        result[f"combined_selected_tenor_{int(tenor)}_count"] = int(
            combined_metrics[f"selected_tenor_{int(tenor)}_count"]
        )

    trades = selected_cols_to_trades(
        combined_selected,
        layer_by_date=layer_by_date,
        candidate_label=candidate_label,
    )

    selected_outputs = {
        "core_selected": core_selected,
        "secondary_raw_selected": secondary_raw_selected,
        "secondary_only_selected": secondary_only_selected,
        "combined_selected": combined_selected,
        "layer_by_date": layer_by_date,
    }

    return result, selected_outputs, trades

In [8]:
# Cell 8 — Evaluate locked 2621 and compare to benchmark

locked_2621_result, locked_2621_selected_outputs, locked_2621_trades = evaluate_core_secondary_candidate(
    LOCKED_2621,
    candidate_label=LOCKED_CANDIDATE_LABEL,
)

locked_2621_summary = pd.DataFrame([locked_2621_result])

summary_display_cols = [
    "candidate",

    "core_trade_count",
    "core_trade_frequency",
    "core_win_rate",

    "secondary_only_trade_count",
    "secondary_only_trade_frequency",
    "secondary_only_win_rate",

    "combined_trade_count",
    "combined_trade_frequency",
    "combined_win_rate",

    "core_share_of_combined",
    "secondary_share_of_combined",

    "combined_avg_pnl_per_day",
    "combined_median_pnl_per_day",
    "combined_aggregate_pnl_per_day",
    "combined_worst_pnl_per_day",
    "combined_avg_selected_tenor",

    "combined_front_count",
    "combined_middle_count",
    "combined_back_count",

    "combined_selected_tenor_9_count",
    "combined_selected_tenor_12_count",
    "combined_selected_tenor_15_count",
    "combined_selected_tenor_18_count",
    "combined_selected_tenor_21_count",
    "combined_selected_tenor_24_count",
    "combined_selected_tenor_27_count",
    "combined_selected_tenor_30_count",
    "combined_selected_tenor_33_count",
]

print("Locked 2621 summary:")
display(locked_2621_summary[summary_display_cols])

# ---------------------------------------------------------------------
# Benchmark drift check
# ---------------------------------------------------------------------
#
# This cell is deliberately non-fatal. It warns if values differ from the
# research benchmark, but does not fail the run. The feature/outcome panel may
# legitimately change when new history is appended.
# ---------------------------------------------------------------------

benchmark_rows = []

for key, expected_value in EXPECTED_2621_BENCHMARK.items():
    actual_value = locked_2621_result.get(key, np.nan)

    benchmark_rows.append({
        "metric": key,
        "expected_research_value": expected_value,
        "actual_current_value": actual_value,
        "difference": actual_value - expected_value if pd.notna(actual_value) else np.nan,
    })

benchmark_check = pd.DataFrame(benchmark_rows)

print("\nBenchmark drift check:")
display(benchmark_check)

max_abs_frequency_or_win_diff = (
    benchmark_check[
        benchmark_check["metric"].str.contains("frequency|win_rate|share", regex=True)
    ]["difference"]
    .abs()
    .max()
)

if pd.notna(max_abs_frequency_or_win_diff) and max_abs_frequency_or_win_diff > 0.0025:
    print(
        "\nWARNING: Current results differ from the locked research benchmark. "
        "This may be expected if upstream panels have been updated."
    )
else:
    print("\nBenchmark check passed or drift is immaterial.")

Locked 2621 summary:


,candidate,core_trade_count,core_trade_frequency,core_win_rate,secondary_only_trade_count,secondary_only_trade_frequency,secondary_only_win_rate,combined_trade_count,combined_trade_frequency,combined_win_rate,core_share_of_combined,secondary_share_of_combined,combined_avg_pnl_per_day,combined_median_pnl_per_day,combined_aggregate_pnl_per_day,combined_worst_pnl_per_day,combined_avg_selected_tenor,combined_front_count,combined_middle_count,combined_back_count,combined_selected_tenor_9_count,combined_selected_tenor_12_count,combined_selected_tenor_15_count,combined_selected_tenor_18_count,combined_selected_tenor_21_count,combined_selected_tenor_24_count,combined_selected_tenor_27_count,combined_selected_tenor_30_count,combined_selected_tenor_33_count
0,locked_core_secondary_2621,386,0.223768,0.865285,191,0.110725,0.780105,577,0.334493,0.837088,0.668977,0.331023,382.748481,508.664099,374.389983,"-5,535.611173",24.639515,179,92,306,53,44,82,30,12,50,10,16,280



Benchmark drift check:


,metric,expected_research_value,actual_current_value,difference
0,combined_trade_count,577.000000,577.000000,0.000000
1,combined_trade_frequency,0.334493,0.334493,-0.000000
2,combined_win_rate,0.837088,0.837088,0.000000
3,core_trade_count,386.000000,386.000000,0.000000
4,core_trade_frequency,0.223768,0.223768,0.000000
5,core_win_rate,0.865285,0.865285,-0.000000
6,secondary_only_trade_count,191.000000,191.000000,0.000000
7,secondary_only_trade_frequency,0.110725,0.110725,-0.000000
8,secondary_only_win_rate,0.780105,0.780105,-0.000000
9,core_share_of_combined,0.668977,0.668977,0.000000



Benchmark check passed or drift is immaterial.


In [9]:
# Cell 9 — Locked 2621 audit tables

# ---------------------------------------------------------------------
# Path-level drawdown and worst clusters
# ---------------------------------------------------------------------

locked_2621_path_summary = pd.DataFrame([
    summarize_trade_frame(
        locked_2621_trades,
        label="Combined",
        denominator=TRADE_FREQUENCY_DENOMINATOR,
    )
])

locked_2621_summary_by_layer = pd.DataFrame(
    [
        summarize_trade_frame(
            df,
            label=layer,
            denominator=TRADE_FREQUENCY_DENOMINATOR,
        )
        for layer, df in locked_2621_trades.groupby("layer", sort=False)
    ]
)

locked_2621_trades["year"] = pd.to_datetime(locked_2621_trades["date"]).dt.year
locked_2621_trades["month"] = pd.to_datetime(locked_2621_trades["date"]).dt.to_period("M").astype(str)

locked_2621_summary_by_year_layer = pd.DataFrame(
    [
        {
            "year": year,
            "layer": layer,
            **summarize_trade_frame(
                df,
                label=f"{year}_{layer}",
                denominator=TRADE_FREQUENCY_DENOMINATOR,
            ),
        }
        for (year, layer), df in locked_2621_trades.groupby(["year", "layer"])
    ]
).drop(columns=["label"], errors="ignore")

locked_2621_summary_by_tenor_layer = pd.DataFrame(
    [
        {
            "selected_tenor": tenor,
            "layer": layer,
            **summarize_trade_frame(
                df,
                label=f"{tenor}_{layer}",
                denominator=TRADE_FREQUENCY_DENOMINATOR,
            ),
        }
        for (tenor, layer), df in locked_2621_trades.groupby(["selected_tenor", "layer"])
    ]
).drop(columns=["label"], errors="ignore")

locked_2621_summary_by_group_layer = pd.DataFrame(
    [
        {
            "tenor_group": tenor_group,
            "layer": layer,
            **summarize_trade_frame(
                df,
                label=f"{tenor_group}_{layer}",
                denominator=TRADE_FREQUENCY_DENOMINATOR,
            ),
        }
        for (tenor_group, layer), df in locked_2621_trades.groupby(["tenor_group", "layer"])
    ]
).drop(columns=["label"], errors="ignore")

locked_2621_worst_trades = (
    locked_2621_trades
    .sort_values("normalized_pnl_dollars")
    .head(50)
    .reset_index(drop=True)
)

locked_2621_worst_20_trade_windows = (
    locked_2621_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl")
    .head(20)
    .loc[
        :,
        [
            "trade_sequence",
            "date",
            "layer",
            "selected_tenor",
            "tenor_group",
            "normalized_pnl_dollars",
            "rolling_20_trade_pnl",
            "cum_pnl",
            "drawdown",
        ],
    ]
    .reset_index(drop=True)
)

print("Locked 2621 path summary:")
display(locked_2621_path_summary)

print("\nSummary by layer:")
display(locked_2621_summary_by_layer)

print("\nSummary by year and layer:")
display(locked_2621_summary_by_year_layer)

print("\nSummary by tenor and layer:")
display(locked_2621_summary_by_tenor_layer)

print("\nSummary by tenor group and layer:")
display(locked_2621_summary_by_group_layer)

print("\nWorst 20 individual trades:")
display(locked_2621_worst_trades.head(20))

print("\nWorst 20-trade rolling windows:")
display(locked_2621_worst_20_trade_windows)

Locked 2621 path summary:


,label,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,Combined,577,0.334493,0.837088,382.748481,508.664099,374.389983,"5,223,114.656812","13,951.000000","-114,955.549452","-5,535.611173","-381,724.761811","-337,212.378098",24.639515



Summary by layer:


,label,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,Secondary,191,0.110725,0.780105,280.257868,456.774878,234.212453,"996,573.985466","4,255.000000","-87,932.829005","-2,958.021794","-249,162.776157","-176,622.419873",22.649215
1,Core,386,0.223768,0.865285,433.462748,549.112179,435.905597,"4,226,540.671347","9,696.000000","-114,955.549452","-5,535.611173","-367,837.180362","-208,249.959008",25.624352



Summary by year and layer:


...

,tenor_group,layer,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,back,Core,224,0.129855,0.897321,443.054716,503.518604,438.725251,"3,167,157.590040","7,219.000000","-114,955.549452","-3,193.209707","-259,853.641796","-162,587.226670",32.558036
1,back,Secondary,82,0.047536,0.780488,197.148408,410.958827,194.766164,"522,947.149951","2,685.000000","-87,932.829005","-2,512.366543","-121,772.219065","-23,191.281047",32.890244
2,front,Core,97,0.056232,0.793814,370.066448,774.832201,375.879961,"442,786.593503","1,178.000000","-60,891.722904","-5,535.611173","-163,985.642996","-90,294.999723",12.402062
3,front,Secondary,82,0.047536,0.768293,364.177082,546.144491,311.620745,"312,867.227575","1,004.000000","-42,928.530585","-2,958.021794","-77,730.661393","37,407.287089",12.585366
4,middle,Core,65,0.037681,0.861538,495.014139,545.792576,474.670121,"616,596.487804","1,299.000000","-49,418.522467","-2,353.262975","-84,727.740892","37,245.576016",21.461538
5,middle,Secondary,27,0.015652,0.814815,277.798613,516.227012,284.027576,"160,759.607940",566.000000,"-34,631.260820","-1,923.958934","-80,026.736764","79,434.352423",22.111111



Worst 20 individual trades:


,candidate,date,layer,selected_col,selected_tenor,tenor_group,spx_close,rv21d,rsi14,vrp_log,vrp_z_3m,vrp_z_1y,tenor_historical_win_probability,win,normalized_pnl_dollars,actual_dte,pnl_per_day,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,locked_core_secondary_2621,2025-02-27,Core,8,33,back,"5,861.570000",11.428641,33.832673,0.939687,1.423970,1.059198,0.793043,0.000000,"-114,955.549452",36,"-3,193.209707",461,"4,163,721.233293","4,472,993.397555","-309,272.164262","-157,867.513537",2025,2025-02
1,locked_core_secondary_2621,2020-01-24,Secondary,8,33,back,"3,295.470000",7.866687,61.573549,1.300539,1.061030,1.738160,0.793043,0.000000,"-87,932.829005",35,"-2,512.366543",54,"127,779.425852","215,712.254857","-87,932.829005","103,142.064304",2020,2020-01
2,locked_core_secondary_2621,2020-02-24,Core,0,9,front,"3,225.890000",18.033963,36.699503,1.425272,0.638828,1.491332,0.731343,0.000000,"-60,891.722904",11,"-5,535.611173",60,"129,975.331506","215,712.254857","-85,736.923351","40,110.552245",2020,2020-02
3,locked_core_secondary_2621,2020-02-27,Core,1,12,front,"2,978.760000",24.489208,20.145213,1.447358,0.664102,1.532049,0.742381,0.000000,"-56,963.300165",15,"-3,797.553344",61,"73,012.031341","215,712.254857","-142,700.223516","-23,088.763889",2020,2020-02
4,locked_core_secondary_2621,2026-02-19,Core,8,33,back,"6,861.890000",12.300219,46.444609,1.045459,0.772548,0.800369,0.793043,0.000000,"-52,092.062100",36,"-1,447.001725",552,"4,918,636.584413","4,994,083.729273","-75,447.144861","-27,810.712165",2026,2026-02
5,locked_core_secondary_2621,2025-02-24,Core,7,30,back,"5,983.250000",11.772753,43.422979,0.796970,1.085260,0.761401,0.788773,0.000000,"-51,336.648143",32,"-1,604.270254",458,"4,372,238.226946","4,472,993.397555","-100,755.170609","98,683.176732",2025,2025-02
6,locked_core_secondary_2621,2025-02-21,Core,5,24,middle,"6,013.130000",11.880124,46.090761,0.741002,0.996426,0.660997,0.767187,0.000000,"-49,418.522467",21,"-2,353.262975",457,"4,423,574.875089","4,472,993.397555","-49,418.522467","165,188.991835",2025,2025-02
7,locked_core_secondary_2621,2026-02-23,Core,8,33,back,"6,837.750000",12.262766,44.737667,1.044061,0.791125,0.791481,0.793043,0.000000,"-48,751.416767",32,"-1,523.481774",554,"4,857,036.210194","4,994,083.729273","-137,047.519079","-104,693.540106",2026,2026-02
8,locked_core_secondary_2621,2025-02-26,Core,8,33,back,"5,956.060000",10.753467,41.127776,0.953633,1.522225,1.104877,0.793043,0.000000,"-47,172.123854",30,"-1,572.404128",460,"4,278,676.782745","4,472,993.397555","-194,316.614811","-26,732.572499",2025,2025-02
9,locked_core_secondary_2621,2025-02-25,Core,8,33,back,"5,955.250000",11.824882,41.026556,0.937331,1.503144,1.069750,0.793043,0.000000,"-46,389.320348",31,"-1,496.429689",459,"4,325,848.906598","4,472,993.397555","-147,144.490957","36,912.396172",2025,2025-02



Worst 20-trade rolling windows:


,trade_sequence,date,layer,selected_tenor,tenor_group,normalized_pnl_dollars,rolling_20_trade_pnl,cum_pnl,drawdown
0,468,2025-03-10,Core,15,front,"15,449.118007","-337,212.378098","4,106,717.753752","-366,275.643804"
1,467,2025-03-07,Core,15,front,"-1,826.626460","-334,298.868342","4,091,268.635744","-381,724.761811"
2,469,2025-03-11,Core,15,front,"21,051.422541","-333,073.294994","4,127,769.176293","-345,224.221262"
3,466,2025-03-06,Core,24,middle,"-6,858.911357","-313,079.000715","4,093,095.262204","-379,898.135351"
4,470,2025-03-13,Secondary,9,front,"13,420.217621","-295,533.413616","4,141,189.393914","-331,804.003641"
5,465,2025-03-05,Secondary,18,middle,"-13,887.581449","-286,624.790126","4,099,954.173561","-373,039.223994"
6,471,2025-04-04,Core,15,front,"31,316.021821","-265,830.324341","4,172,505.415735","-300,487.981820"
7,464,2025-03-04,Core,21,middle,"-14,340.230004","-256,912.437062","4,113,841.755010","-359,151.642545"
8,472,2025-04-07,Core,21,middle,"36,248.703640","-227,648.817709","4,208,754.119375","-264,239.278181"
9,463,2025-03-03,Core,21,middle,"-14,110.077064","-226,559.643881","4,128,181.985014","-344,811.412541"


In [10]:
# Cell 10 — Save locked 2621 outputs

LOCKED_2621_SELECTED_TRADES_PARQUET = (
    PROCESSED_DIR / "simple_signal_grid_locked_2621_selected_trades_v0_1.parquet"
)

LOCKED_2621_SELECTED_TRADES_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_selected_trades_v0_1.csv"
)

LOCKED_2621_SUMMARY_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_v0_1.csv"
)

LOCKED_2621_PARAMETERS_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_parameters_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_layer_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_YEAR_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_year_layer_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_TENOR_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_tenor_layer_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_GROUP_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_group_layer_v0_1.csv"
)

LOCKED_2621_WORST_TRADES_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_worst_trades_v0_1.csv"
)

LOCKED_2621_WORST_20_WINDOWS_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_worst_20_trade_windows_v0_1.csv"
)

LOCKED_2621_SUMMARY_JSON = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_v0_1.json"
)

# Save trade-level path.
locked_2621_trades.to_parquet(LOCKED_2621_SELECTED_TRADES_PARQUET, index=False)
locked_2621_trades.to_csv(LOCKED_2621_SELECTED_TRADES_CSV, index=False)

# Save summary/audit tables.
locked_2621_summary.to_csv(LOCKED_2621_SUMMARY_CSV, index=False)
locked_2621_parameter_table.to_csv(LOCKED_2621_PARAMETERS_CSV, index=False)
locked_2621_summary_by_layer.to_csv(LOCKED_2621_SUMMARY_BY_LAYER_CSV, index=False)
locked_2621_summary_by_year_layer.to_csv(LOCKED_2621_SUMMARY_BY_YEAR_LAYER_CSV, index=False)
locked_2621_summary_by_tenor_layer.to_csv(LOCKED_2621_SUMMARY_BY_TENOR_LAYER_CSV, index=False)
locked_2621_summary_by_group_layer.to_csv(LOCKED_2621_SUMMARY_BY_GROUP_LAYER_CSV, index=False)
locked_2621_worst_trades.to_csv(LOCKED_2621_WORST_TRADES_CSV, index=False)
locked_2621_worst_20_trade_windows.to_csv(LOCKED_2621_WORST_20_WINDOWS_CSV, index=False)

summary_payload = {
    "candidate_label": LOCKED_CANDIDATE_LABEL,
    "project_root": str(PROJECT_ROOT),
    "feature_panel_path": str(FEATURE_PANEL_PATH),
    "outcome_path": str(OUTCOME_PATH),
    "trade_frequency_denominator": int(TRADE_FREQUENCY_DENOMINATOR),
    "sweep_start_date": str(sweep_dates.min().date()),
    "sweep_end_date": str(sweep_dates.max().date()),
    "parameters": LOCKED_2621,
    "summary": locked_2621_result,
    "expected_research_benchmark": EXPECTED_2621_BENCHMARK,
}

with open(LOCKED_2621_SUMMARY_JSON, "w") as f:
    json.dump(summary_payload, f, indent=2, default=str)

print("Saved locked 2621 outputs:")
print("  ", LOCKED_2621_SELECTED_TRADES_PARQUET)
print("  ", LOCKED_2621_SELECTED_TRADES_CSV)
print("  ", LOCKED_2621_SUMMARY_CSV)
print("  ", LOCKED_2621_PARAMETERS_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_LAYER_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_YEAR_LAYER_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_TENOR_LAYER_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_GROUP_LAYER_CSV)
print("  ", LOCKED_2621_WORST_TRADES_CSV)
print("  ", LOCKED_2621_WORST_20_WINDOWS_CSV)
print("  ", LOCKED_2621_SUMMARY_JSON)

Saved locked 2621 outputs:
   C:\Users\patri\vrp_project\data\processed\simple_signal_grid_locked_2621_selected_trades_v0_1.parquet
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_selected_trades_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_parameters_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_year_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_tenor_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_group_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_worst_trades_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_worst_20_trade_

## Future testing guide

This notebook is now the **locked 2621 baseline**. Future tests should be incremental and auditable.

Recommended next tests:

1. **Sizing tests**
   - Core = 1.00x
   - Secondary = 0.50x / 0.75x / 1.00x
   - Compare weighted P&L, max drawdown, and worst rolling trade clusters.

2. **Validation tests**
   - Year-by-year Core vs Secondary.
   - Tenor-level attribution.
   - Worst cluster review.
   - Stress behavior around high-volatility regimes.

3. **New parameter tests**
   - Copy `LOCKED_2621`.
   - Modify one small group of thresholds.
   - Re-run `evaluate_core_secondary_candidate()`.
   - Compare against `locked_2621_summary`.

Avoid reintroducing broad grid sweeps into this notebook. If broad searches are needed again, create a separate research notebook and bring only the winning candidate back here.

In [11]:
# Cell 11 — Optional future-test template
#
# Leave RUN_EXAMPLE_VARIANT_TEST = False for normal locked-candidate runs.
# To test a small incremental change, copy LOCKED_2621, edit one or two values,
# and compare the result to locked_2621_summary.

RUN_EXAMPLE_VARIANT_TEST = False

if RUN_EXAMPLE_VARIANT_TEST:
    test_candidate = deepcopy(LOCKED_2621)
    test_candidate["candidate_label"] = "example_variant"

    # Example edit:
    # test_candidate["secondary"]["front"]["rsi_cap"] = 73.0

    test_result, test_selected_outputs, test_trades = evaluate_core_secondary_candidate(
        test_candidate,
        candidate_label=test_candidate["candidate_label"],
    )

    comparison = pd.concat(
        [
            locked_2621_summary.assign(test_case="locked_2621"),
            pd.DataFrame([test_result]).assign(test_case="example_variant"),
        ],
        ignore_index=True,
    )

    display(comparison[["test_case"] + summary_display_cols[1:]])

## End of cleaned notebook

The executable notebook stops here.

Archived research that led to 2621:

- Pair 144 established the Core/Secondary structure.
- Relaxing front/middle z-score equality produced pair 2621.
- 90% Core target was not attainable with the tested monotonic rules.
- Full tenor-line fitting underperformed.
- Hybrid VRP/RSI line smoothing also underperformed.
- Therefore, the bucketed 2621 rule is the locked candidate for this phase.